# Tensor manipulation using einops

In this notebook we will use einops to rewrite typical deep learning operations in a more clear and concise way. First, let's intall einops:

In [ ]:
!pip install einops

import torch
import einops
from einops import rearrange, reduce, einsum

## Flattening a tensor

This operation is used typically before fully connected layers.

The size of input tensor will be b c h w (batch of images). And the size ouput should be b (c h w)

In [ ]:
x = torch.randn(4,3,6,7)

# write the einops operation here
y = rearrange(x, 'b c h w -> b (c h w)')

print(y.shape)

torch.Size([4, 126])


## Pooling

Pooling is typically used to reduce the spatial size of a tensor in convolutional networks.

Rewrite pooling2d using einops:

In [ ]:
x = torch.randn(4,3,10,10)

import  torch.nn.functional as F

y = F.avg_pool2d(x,(2,2),2)
print(y.shape)

#repeat average pooling using einops
y2 = reduce(x, 'b c (h h1) (w w1) -> b c h w', 'mean', h1=2, w1=2)
print(y2.shape)

print("the result of difference should be close to zero:")
print((y-y2).abs().max())


torch.Size([4, 3, 5, 5])
torch.Size([4, 3, 5, 5])
the result of difference should be close to zero:
tensor(1.1921e-07)


## Patch embedding

Patch embedding is the first operation in vision transformers. The steps are:

1. Split image into patches
2. Flatten patches
3. Apply a linear transformation

In many implementations, patch embedding is done using a convolutional layer as follows:

In [ ]:
from torch import nn
from torch import Tensor

class PatchEmbedding(nn.Module):
    def __init__(self, in_channels: int = 3, patch_size: int = 16, emb_size: int = 768):
        self.patch_size = patch_size
        super().__init__()
        self.projection =  nn.Conv2d(in_channels, emb_size, kernel_size=patch_size, stride=patch_size)


    def forward(self, x: Tensor) -> Tensor:
        x = self.projection(x)
        # reshape from b embed_size h//patch_size w//patch_size to b ( h//patch_size w//patch_size) embed_size
        x = x.view(x.shape[0], x.shape[1], -1) # these two lines can be done with einops too!
        x = x.transpose(-2,-1)


        return x

In [ ]:
x = torch.randn(4,3,224,224)

pe = PatchEmbedding(3,16,768)

y = pe(x)

print(y.shape)


torch.Size([4, 196, 768])


In [ ]:
from einops.layers.torch import Rearrange

class PatchEmbedding_einops(nn.Module):
    def __init__(self, in_channels: int = 3, patch_size: int = 16, emb_size: int = 768):
        self.patch_size = patch_size
        super().__init__()
        #extract and flatten patches using Rearrange Layer
        self.rearrange = Rearrange('b c (h h1) (w w1) -> b (h w) (c h1 w1)',  h1=patch_size, w1=patch_size )
        self.projection =  nn.Linear(in_channels * patch_size * patch_size, emb_size)


    def forward(self, x: Tensor) -> Tensor:
        x = self.rearrange(x)
        x = self.projection(x)

        return x

In [ ]:
x = torch.randn(4,3,224,224)

pe_einops = PatchEmbedding_einops(3,16,768)

y = pe_einops(x)

print(y.shape)

torch.Size([4, 196, 768])


# Self attention

Lets implement a self attention block using basic torch operations


In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, input_dim: int = 768, dropout: float = 0):
        super().__init__()
        self.input_dim = input_dim
        # fuse the queries, keys and values in one matrix (more efficient)
        self.qkv = nn.Linear(input_dim, input_dim * 3)
        self.att_drop = nn.Dropout(dropout)
        self.projection = nn.Linear(input_dim, input_dim)

    def forward(self, x : Tensor, mask: Tensor = None) -> Tensor:
        x_qkv =self.qkv(x)
        # split keys, queries and values from x_qkv
        queries, keys, values = torch.chunk(x_qkv,3, dim=-1)

        print('values shape:', values.shape)

        scaling = self.input_dim ** (1/2)
        energy = torch.bmm(queries, keys.transpose(-1,-2)) #bactched matrix multiplication
        print('energy shape:', energy.shape)
        if mask is not None:
          fill_value = torch.finfo(torch.float32).min
          energy.mask_fill(~mask, fill_value)

        att = F.softmax(energy, dim = -1) / scaling
        att = self.att_drop(att)

        out = torch.bmm(att, values)  #bactched matrix multiplication

        return out


x = torch.randn(5,100,768)
sa = SelfAttention(768,0.0)

y = sa(x)
print(y.shape)




values shape: torch.Size([5, 100, 768])
energy shape: torch.Size([5, 100, 100])
torch.Size([5, 100, 768])


Now rewrite SelfAttention using einops

In [ ]:

class SelfAttention_einops(nn.Module):
    def __init__(self, input_dim: int = 768, dropout: float = 0):
        super().__init__()
        self.input_dim = input_dim
        # fuse the queries, keys and values in one matrix (more efficient)
        self.qkv = nn.Linear(input_dim, input_dim * 3)
        self.att_drop = nn.Dropout(dropout)
        self.projection = nn.Linear(input_dim, input_dim)

    def forward(self, x : Tensor, mask: Tensor = None) -> Tensor:
        x_qkv =self.qkv(x)
        # split keys, queries and values from x_qkv
        # Reshape to 3 b n embed_dim using einops
        x_qkv = rearrange(x_qkv, 'h w (x e) -> x h w e', x=3)

        queries, keys, values = x_qkv[0], x_qkv[1], x_qkv[2]

        print('values shape:', values.shape)

        scaling = self.input_dim ** (1/2)
        # calculate energy using einsum
        energy =  torch.einsum('ijk, ilk -> ijl', queries, keys)
        print('energy shape:', energy.shape)
        if mask is not None:
          fill_value = torch.finfo(torch.float32).min
          energy.mask_fill(mask==1, fill_value)

        att = F.softmax(energy, dim = -1) / scaling
        att = self.att_drop(att)

        #multiply att with values using einops
        out = torch.einsum('ijk, ijl -> ikl', att, values)
        return out


x = torch.randn(5,100,768)
sa = SelfAttention_einops(768,0.0)

y = sa(x)
print(y.shape)

values shape: torch.Size([5, 100, 768])
energy shape: torch.Size([5, 100, 100])
torch.Size([5, 100, 768])


## Multihead attention

Now let's write multihead attention

In [ ]:
import torch
import torch.nn as nn

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, input_dim, num_heads, dropout=0.0):
        super(MultiHeadSelfAttention, self).__init__()
        assert input_dim % num_heads == 0, "Input dimension must be divisible by the number of heads"

        self.input_dim = input_dim
        self.num_heads = num_heads
        self.head_dim = input_dim // num_heads

        self.qkv = nn.Linear(input_dim, 3*input_dim)


        self.dropout = nn.Dropout(dropout)
        self.output_projection = nn.Linear(input_dim, input_dim)

    def forward(self, x, mask=None):
        batch_size, seq_len, input_dim = x.size()


        x_qkv = self.qkv(x) # only one linear layer
                            # x_qkv contains queries, keys values for all heads
        print('initial shape:', x_qkv.shape)
        # split in querys, keys, values
        queries, keys, values = torch.chunk(x_qkv,3, dim=-1)
        print('queries shape:', queries.shape)

        # Reshape queries, keys, and values to split heads, the ouput dim is:  b num_heads seq_len head_dim
        queries = queries.view(batch_size, seq_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        keys = keys.view(batch_size, seq_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        values = values.view(batch_size, seq_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        print('queries shape after head split: ', queries.shape)

        # Compute scaled dot-product attention
        scaling = self.input_dim ** (1/2)
        energy = torch.matmul(queries, keys.transpose(-2, -1)) / scaling
        print('energy shape: ', energy.shape)
        if mask is not None:
            fill_value = torch.finfo(torch.float32).min
            energy.mask_fill(~mask, fill_value)

        att = F.softmax(energy, dim=-1)
        att = self.dropout(att)

        # Apply attention to values
        out = torch.matmul(att, values)
        print('attn res shape: ', out.shape)

        #importat detail: contigous is required before view (or use reshape!)
        out = out.permute(0, 2, 1, 3).contiguous().view(batch_size, seq_len, input_dim)


        return out


x = torch.randn(5,100,768)
msa = MultiHeadSelfAttention(768,12)

y = msa(x)
print(y.shape)


initial shape: torch.Size([5, 100, 2304])
queries shape: torch.Size([5, 100, 768])
queries shape after head split:  torch.Size([5, 12, 100, 64])
energy shape:  torch.Size([5, 12, 100, 100])
attn res shape:  torch.Size([5, 12, 100, 64])
torch.Size([5, 100, 768])


Now reimplement the previous class using einops

In [ ]:
import torch
import torch.nn as nn

class MultiHeadSelfAttention_einops(nn.Module):
    def __init__(self, input_dim, num_heads, dropout=0.0):
        super(MultiHeadSelfAttention_einops, self).__init__()
        assert input_dim % num_heads == 0, "Input dimension must be divisible by the number of heads"

        self.input_dim = input_dim
        self.num_heads = num_heads
        self.head_dim = input_dim // num_heads

        self.qkv = nn.Linear(input_dim, 3*input_dim)


        self.dropout = nn.Dropout(dropout)
        self.output_projection = nn.Linear(input_dim, input_dim)

    def forward(self, x, mask=None):
        batch_size, seq_len, input_dim = x.size()

        x_qkv = self.qkv(x) # only one linear layer
                            # x_qkv contains queries, keys values for all heads
        print('initial shape:', x_qkv.shape)

        # split in querys, keys, values using rearrange as before
        x_qkv = rearrange(x_qkv, 'h w (x e) -> x h w e', x=3)

        queries, keys, values = x_qkv[0], x_qkv[1], x_qkv[2]
        print('queries shape:', queries.shape)



        # Reshape queries, keys, and values to split heads, the ouput dim is:  b num_heads seq_len head_dim
        queries = rearrange(queries, 'b sl (nh hd)-> b nh sl hd', nh=self.num_heads, hd=self.head_dim)
        keys = rearrange(keys, 'b sl (nh hd)-> b nh sl hd', nh=self.num_heads, hd=self.head_dim)
        values = rearrange(values, 'b sl (nh hd)-> b nh sl hd', nh=self.num_heads, hd=self.head_dim)
        print('queries shape after head split: ', queries.shape)

        # Compute scaled dot-product attention usin einops
        scaling = self.input_dim ** (1/2)
        energy = torch.einsum('ijlk, ijmk -> ijlm', queries, keys)/scaling
        print('energy shape: ', energy.shape)
        if mask is not None:
            fill_value = torch.finfo(torch.float32).min
            energy.mask_fill(~mask, fill_value)

        att = F.softmax(energy, dim=-1)
        att = self.dropout(att)

        # Apply attention to values usin einops
        out = torch.einsum('ijkl, ijkm -> ijkm', queries, keys)
        print('attn res shape: ', out.shape)

        # rearrange out to join heads
        out = rearrange(out, 'b nh sl hd -> b sl (nh hd)', nh=self.num_heads, hd=self.head_dim)



        return out


x = torch.randn(5,100,768)
msa = MultiHeadSelfAttention_einops(768,12)

y = msa(x)
print(y.shape)


initial shape: torch.Size([5, 100, 2304])
queries shape: torch.Size([5, 100, 768])
queries shape after head split:  torch.Size([5, 12, 100, 64])
energy shape:  torch.Size([5, 12, 100, 100])
attn res shape:  torch.Size([5, 12, 100, 64])
torch.Size([5, 100, 768])
